# Model Training — Baseline Run

Baseline using raw features only (no engineering) to establish a comparison point before adding engineered features.

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, RocCurveDisplay,
    PrecisionRecallDisplay,
)
import xgboost as xgb

from src.data import load_config, load_raw, clean, split

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

cfg = load_config("../config.yaml")

## 2. Load & Prepare Data

In [ ]:
df = clean(load_raw("../data/raw"))
train_df, test_df = split(df, test_size=cfg["data"]["test_size"], random_state=cfg["split"]["random_state"])

TARGET   = cfg["data"]["target_col"]
DROP_COLS = ["id", TARGET]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df[TARGET]

print(f"Features: {X_train.columns.tolist()}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 3. Baseline Models

In [ ]:
def evaluate(name, y_true, probs):
    """Return PR-AUC and ROC-AUC for a model."""
    return {
        "model":   name,
        "pr_auc":  round(average_precision_score(y_true, probs), 4),
        "roc_auc": round(roc_auc_score(y_true, probs), 4),
    }

results = []

# --- Logistic Regression ---
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    )),
])
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
results.append(evaluate("Logistic Regression", y_test, lr_probs))
print("Logistic Regression done")

# --- XGBoost (default params, class imbalance handled) ---
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train, y_train)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
results.append(evaluate("XGBoost (baseline)", y_test, xgb_probs))
print("XGBoost done")

results_df = pd.DataFrame(results).set_index("model")
results_df

## 4. Precision-Recall & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models = [
    ("Logistic Regression", lr_probs),
    ("XGBoost (baseline)",  xgb_probs),
]

for name, probs in models:
    PrecisionRecallDisplay.from_predictions(
        y_test, probs, name=name, ax=axes[0]
    )
    RocCurveDisplay.from_predictions(
        y_test, probs, name=name, ax=axes[1]
    )

# Baseline: random classifier PR line
axes[0].axhline(y_test.mean(), color="grey", linestyle="--", label=f"Random ({y_test.mean():.2f})")
axes[0].set_title("Precision-Recall Curve")
axes[1].set_title("ROC Curve")
axes[0].legend()
axes[1].legend()
plt.tight_layout()

## 5. XGBoost Feature Importance

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(8, 5))
colors = ["#DD8452" if v == importances.max() else "#4C72B0" for v in importances.values]
plt.barh(importances.index, importances.values, color=colors)
plt.title("XGBoost Feature Importance (baseline)")
plt.xlabel("Importance score")
plt.tight_layout()

## 6. Baseline Summary

In [ ]:
print("=" * 45)
print("BASELINE RESULTS (raw features, no tuning)")
print("=" * 45)
print(results_df.to_string())
print()
print("Next steps:")
print("  1. Implement features.py (interactions, freq encoding, log transform)")
print("  2. Re-run with engineered features — measure PR-AUC lift")
print("  3. Tune XGBoost with Optuna")
print("  4. Add SHAP analysis")